In [6]:
import os
print(os.getcwd()) ##set wd

C:\Users\Antonio\anaconda_projects\ef4ef301-aa4b-456a-8296-60c9e31af76d


In [1]:
import random
import numpy as np

weights = [5, 8, 6, 3, 7, 15, 6, 18, 20, 6, 10, 12, 4, 7, 15] #art supplies costs
values = [9, 7, 8, 6, 8, 9, 7, 9, 8, 7, 8, 8, 5, 6, 7] #usefulness score
cap = 100 #buget
n = len(weights)

This cell sets up the data for the model. I import the random and numpy library, define price (weights) and usefulness scores (values) for each 15 products. I also set the budget as cap = 100 and store the number in items in n. 

In [2]:
# Random Search #
steps_RS = 1000

best_sol_RS = np.zeros(n, dtype=int)
best_val_RS = 0

for _ in range(steps_RS):
    new_sol = np.zeros(n, dtype=int)
    new_weight = 0
    new_value = 0

    for i in range(n):
        if random.random() < 0.5:
            new_sol[i] = 1
            new_weight += weights[i]
            if new_weight > cap:
                new_sol[i] = 0
                new_weight -= weights[i]

    for i in range(n):
        new_value += values[i] * new_sol[i]

    if new_value > best_val_RS:
        best_val_RS = new_value
        best_sol_RS = new_sol.copy()

print("RS best solution:", best_sol_RS.tolist(), "value:", best_val_RS,
      "cost:", sum(weights[i] for i in range(n) if best_sol_RS[i] == 1))

RS best solution: [1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0] value: 86 cost: 96


The above cell runs a random search for 1 million iterations. In each iteration it builds a random feasible 0 and 1 selecton under £100 buget, evaluate the total usefulness score and stores the best solution.


In [3]:
# Hill Climb #
steps_HC = 1000000

best_value_HC = 0
sol_invalid = True
while sol_invalid:
    best_sol_HC = []
    best_weight_HC = 0
    for i in range(n):
        best_sol_HC.append(random.randint(0, 1))
    for i in range(n):
        best_weight_HC += weights[i] * best_sol_HC[i]
    sol_invalid = (best_weight_HC > cap)

for i in range(n):
    best_value_HC += values[i] * best_sol_HC[i]

# HC loop
for _ in range(1, steps_HC):
    new_value = 0
    sol_invalid = True
    while sol_invalid:
        new_sol = best_sol_HC.copy()
        new_weight = 0

        entry = random.randint(0, n-1)        # flip
        new_sol[entry] = 1 - new_sol[entry]

        for i in range(n):
            new_weight += weights[i] * new_sol[i]
        sol_invalid = (new_weight > cap)

    for i in range(n):
        new_value += values[i] * new_sol[i]

    if new_value > best_value_HC:
        best_sol_HC = new_sol.copy()
        best_value_HC = new_value

print("HC best solution:", best_sol_HC, "value:", best_value_HC,
      "cost:", sum(weights[i] for i in range(n) if best_sol_HC[i] == 1))

HC best solution: [0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1] value: 77 cost: 98


The above code applies a hill climb. It generates a random 0 / 1 start under the budget set, then for any iteratons it flips - keeps it if the usefulness improves or discards it goes over.

In [4]:
# Dynamic Programming #

max_weight = cap
items = [(weights[i], values[i]) for i in range(n)]

dp_table = [[0 for _ in range(max_weight + 1)] for _ in range(n + 1)]

#table for optimal values for weight and item combo
for i in range(1, n + 1):
    weight, value = items[i - 1]
    for w in range(1, max_weight + 1):
        if weight > w:
            dp_table[i][w] = dp_table[i - 1][w]
        else:
            dp_table[i][w] = max(dp_table[i - 1][w],
                                 dp_table[i - 1][w - weight] + value)

total_value = dp_table[-1][-1]

# go back to find selected items
selected_items = []
w = max_weight
for i in range(n, 0, -1):
    if dp_table[i][w] != dp_table[i - 1][w]:
        selected_items.insert(0, i)
        w -= items[i - 1][0]

print("DP total value:", total_value)
print("DP items (1-based):", selected_items)
print("DP total cost:", sum(weights[i-1] for i in selected_items))

DP total value: 91
DP items (1-based): [1, 2, 3, 4, 5, 6, 7, 8, 10, 11, 12, 13]
DP total cost: 100


Table DP wil the best usefulness using the first i and w, but goes back to pick the optimal items for the best bundle